# GGUF Format and CPU Inference with llama.cpp

**Stage 4: Advanced Optimization - Notebook 43**

This notebook explores GGUF (GPT-Generated Unified Format) and CPU inference using llama.cpp, enabling efficient LLM inference on consumer hardware without GPUs. We'll cover:
- GGUF format explained (successor to GGML)
- Why GGUF for CPU inference
- Different quantization levels (Q4_0, Q5_0, Q8_0, etc.)
- Installing and using llama.cpp
- Converting models to GGUF format
- CPU performance benchmarks
- Apple Silicon (Metal) optimization
- When CPU inference makes sense

**Related notebooks**: 
- `41_gptq_quantization.ipynb` - GPU quantization (GPTQ)
- `42_awq_quantization.ipynb` - GPU quantization (AWQ)
- `56_speculative_decoding.ipynb` - Speed optimization

**Reference**: [LLM_INFERENCE_ROADMAP.md](../LLM_INFERENCE_ROADMAP.md) - Stage 4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import platform
import subprocess
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# System info
print("System Information")
print("="*60)
print(f"OS: {platform.system()} {platform.release()}")
print(f"Architecture: {platform.machine()}")
print(f"Processor: {platform.processor()}")

# Check for Apple Silicon
is_apple_silicon = platform.system() == 'Darwin' and platform.machine() == 'arm64'
if is_apple_silicon:
    print("\n✓ Apple Silicon detected - Metal acceleration available!")
else:
    print("\n  CPU-only inference (no Metal)")

## 1. GGUF Format Explained

### What is GGUF?

**GGUF (GPT-Generated Unified Format)**:
- File format for storing LLMs optimized for CPU/local inference
- Successor to GGML format (deprecated)
- Created by Georgi Gerganov (llama.cpp author)
- Released: August 2023

### Why GGUF?

**Problems with Standard Formats**:
```
PyTorch (.pth, .safetensors):
- Optimized for GPU inference
- Large file size
- Slow loading on CPU
- No built-in quantization metadata

ONNX:
- Better than PyTorch for CPU
- Still not optimized for quantization
- Limited quantization options
```

**GGUF Advantages**:
```
1. Memory-mapped loading:
   - Instant startup (no model loading wait)
   - OS manages memory (can use more than RAM)
   - Multiple processes can share same file

2. Native quantization support:
   - Q4_0, Q4_1, Q5_0, Q5_1, Q8_0, etc.
   - Metadata for each quantization level
   - Optimized for CPU/SIMD instructions

3. Extensible format:
   - Can add new quantization methods
   - Model metadata embedded
   - Version control built-in

4. Cross-platform:
   - Works on x86, ARM, Apple Silicon
   - No Python dependencies
   - C++ implementation (fast)
```

### Historical Context

**Timeline**:
- **March 2023**: llama.cpp released (GGML format)
- **May 2023**: Explosion of community interest (run Llama on laptop!)
- **August 2023**: GGUF format introduced (improved GGML)
- **2024-Present**: GGUF becomes standard for local LLM inference

**Impact**:
```
Before llama.cpp/GGUF:
- LLMs required expensive GPUs
- Consumer hardware couldn't run 7B+ models
- Local AI was impractical

After llama.cpp/GGUF:
- 7B models run on laptops (5-20 tokens/sec)
- 13B models run on desktops
- 70B models run on high-end workstations
- Democratized AI access
```

In [ ]:
# Visualize GGUF advantages
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Loading time comparison
ax = axes[0, 0]
formats = ['PyTorch', 'ONNX', 'GGUF']
load_times = [12.5, 8.2, 0.1]  # seconds for Llama 2 7B
colors = ['#e74c3c', '#f39c12', '#27ae60']

bars = ax.bar(formats, load_times, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Load Time (seconds)', fontsize=12)
ax.set_title('Model Loading Time: Llama 2 7B', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_yscale('log')

# Add labels
for bar, time in zip(bars, load_times):
    if time < 1:
        label = f'{time*1000:.0f}ms'
    else:
        label = f'{time:.1f}s'
    ax.text(bar.get_x() + bar.get_width()/2, time, label,
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# 2. File size comparison (different quantizations)
ax = axes[0, 1]
quant_levels = ['FP16', 'Q8_0', 'Q5_1', 'Q4_1', 'Q4_0', 'Q2_K']
sizes_gb = [13.5, 7.2, 4.6, 3.9, 3.6, 2.5]  # Llama 2 7B
quality_loss = [0, 0.1, 0.3, 0.5, 0.8, 3.0]  # percentage points

# Create color gradient based on quality
colors_quant = plt.cm.RdYlGn_r([q / 3.5 for q in quality_loss])

bars = ax.barh(quant_levels, sizes_gb, color=colors_quant, alpha=0.7, edgecolor='black')
ax.set_xlabel('File Size (GB)', fontsize=12)
ax.set_title('GGUF Quantization Levels (Llama 2 7B)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add quality loss labels
for bar, size, loss in zip(bars, sizes_gb, quality_loss):
    label = f'{size:.1f}GB ({loss:.1f}% loss)'
    ax.text(size + 0.3, bar.get_y() + bar.get_height()/2, label,
            va='center', fontsize=9, fontweight='bold')

# 3. Hardware compatibility
ax = axes[1, 0]
hardware = ['M1 MacBook\n8GB', 'Desktop\n16GB', 'Desktop\n32GB', 'Workstation\n64GB']
max_model = ['7B Q4', '13B Q4', '33B Q4', '70B Q4']
colors_hw = ['#3498db', '#9b59b6', '#e67e22', '#e74c3c']

bars = ax.barh(hardware, [8, 16, 32, 64], color=colors_hw, alpha=0.7, edgecolor='black')
ax.set_xlabel('RAM (GB)', fontsize=12)
ax.set_title('Maximum Model Size (GGUF Q4)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add model size labels
for bar, model in zip(bars, max_model):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2, model,
            va='center', fontsize=11, fontweight='bold')

# 4. Speed comparison (tokens/sec)
ax = axes[1, 1]
platforms = ['Intel i7', 'AMD Ryzen 9', 'M1 Pro', 'M2 Max', 'M3 Max']
speeds = [8, 12, 18, 28, 42]  # tokens/sec for Llama 2 7B Q4
colors_platform = ['#3498db', '#e74c3c', '#95a5a6', '#7f8c8d', '#2c3e50']

bars = ax.barh(platforms, speeds, color=colors_platform, alpha=0.7, edgecolor='black')
ax.set_xlabel('Speed (tokens/sec)', fontsize=12)
ax.set_title('CPU Inference Speed (Llama 2 7B Q4_0)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add speed labels
for bar, speed in zip(bars, speeds):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{speed} tok/s',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("1. GGUF loads 100x faster than PyTorch (memory-mapped)")
print("2. Q4_0 achieves 3.8x compression with <1% quality loss")
print("3. 7B models run on consumer hardware (8GB+ RAM)")
print("4. Apple Silicon 2-5x faster than x86 CPUs (unified memory + Metal)")

## 2. GGUF Quantization Levels

### Understanding Quantization Naming

**Format**: `Q{bits}_{variant}`
- `Q`: Quantized
- `{bits}`: Number of bits (2, 3, 4, 5, 6, 8)
- `{variant}`: Quantization method (0, 1, K_S, K_M, K_L)

### Common Quantization Levels

**Q2_K** (2-bit):
```
Size: ~2.5GB (7B model)
Quality: Significant loss (3-5%)
Speed: Very fast
Use case: Extreme compression, draft models
```

**Q3_K_M** (3-bit medium):
```
Size: ~3.0GB (7B model)
Quality: Noticeable loss (1.5-2.5%)
Speed: Very fast
Use case: Balanced compression/quality
```

**Q4_0** (4-bit, variant 0):
```
Size: ~3.6GB (7B model)
Quality: Minimal loss (0.5-1%)
Speed: Fast
Use case: DEFAULT CHOICE - best balance
Recommended: Start here!
```

**Q4_K_M** (4-bit K-quant medium):
```
Size: ~4.1GB (7B model)
Quality: Better than Q4_0 (0.3-0.8%)
Speed: Fast
Use case: Higher quality at 4-bit
```

**Q5_0** (5-bit):
```
Size: ~4.6GB (7B model)
Quality: Very good (0.2-0.5%)
Speed: Medium-fast
Use case: Quality-sensitive applications
```

**Q5_K_M** (5-bit K-quant medium):
```
Size: ~4.8GB (7B model)
Quality: Excellent (0.1-0.3%)
Speed: Medium-fast
Use case: Maximum quality at reasonable size
```

**Q8_0** (8-bit):
```
Size: ~7.2GB (7B model)
Quality: Near-perfect (<0.1%)
Speed: Medium
Use case: Quality > size, or when 8-bit is sufficient
```

**F16** (FP16 - no quantization):
```
Size: ~13.5GB (7B model)
Quality: Perfect (baseline)
Speed: Slower (more memory bandwidth)
Use case: Maximum quality, plenty of RAM
```

### K-Quants Explained

**K-quant variants** (`_K_S`, `_K_M`, `_K_L`):
```
_K_S (Small):
- Most aggressive quantization
- Smallest size
- Lowest quality

_K_M (Medium): ✓ RECOMMENDED
- Balanced approach
- Good size/quality trade-off
- Best for most use cases

_K_L (Large):
- Less aggressive quantization
- Larger size
- Highest quality

K-quant advantages:
- Per-layer quantization (some layers higher precision)
- Attention layers often get more bits
- Better quality than non-K at same average bits
```

In [ ]:
# Detailed quantization comparison
import pandas as pd

print("GGUF Quantization Level Guide (Llama 2 7B)")
print("="*90)

quant_data = [
    {'Level': 'F16', 'Size_GB': 13.5, 'Quality_Loss': 0.0, 'Speed_Rel': 70, 'RAM_Needed': '16GB+', 'Recommended': 'Max quality'},
    {'Level': 'Q8_0', 'Size_GB': 7.2, 'Quality_Loss': 0.1, 'Speed_Rel': 85, 'RAM_Needed': '10GB+', 'Recommended': 'Very high quality'},
    {'Level': 'Q6_K', 'Size_GB': 5.5, 'Quality_Loss': 0.2, 'Speed_Rel': 90, 'RAM_Needed': '8GB+', 'Recommended': 'High quality'},
    {'Level': 'Q5_K_M', 'Size_GB': 4.8, 'Quality_Loss': 0.3, 'Speed_Rel': 95, 'RAM_Needed': '8GB+', 'Recommended': 'Excellent balance'},
    {'Level': 'Q5_0', 'Size_GB': 4.6, 'Quality_Loss': 0.4, 'Speed_Rel': 95, 'RAM_Needed': '6GB+', 'Recommended': 'Good quality'},
    {'Level': 'Q4_K_M', 'Size_GB': 4.1, 'Quality_Loss': 0.6, 'Speed_Rel': 100, 'RAM_Needed': '6GB+', 'Recommended': 'Best default'},
    {'Level': 'Q4_0', 'Size_GB': 3.6, 'Quality_Loss': 0.8, 'Speed_Rel': 100, 'RAM_Needed': '6GB+', 'Recommended': 'Fast & small'},
    {'Level': 'Q3_K_M', 'Size_GB': 3.0, 'Quality_Loss': 2.0, 'Speed_Rel': 105, 'RAM_Needed': '5GB+', 'Recommended': 'Aggressive compression'},
    {'Level': 'Q2_K', 'Size_GB': 2.5, 'Quality_Loss': 4.0, 'Speed_Rel': 110, 'RAM_Needed': '4GB+', 'Recommended': 'Extreme compression'},
]

df = pd.DataFrame(quant_data)
print(df.to_string(index=False))

print("\n" + "="*90)
print("\nRecommendations:")
print("  • Start with Q4_K_M (best default)")
print("  • Quality-sensitive → Q5_K_M or Q8_0")
print("  • Memory-constrained → Q4_0 or Q3_K_M")
print("  • Maximum quality → F16 (if RAM permits)")
print("\nQuality Loss Legend:")
print("  <1% = Imperceptible, 1-2% = Acceptable, >2% = Noticeable")

## 3. Installing and Using llama.cpp

### Installation

**Option 1: Pre-built binaries** (easiest):
```bash
# macOS (Homebrew)
brew install llama.cpp

# Or download from releases:
# https://github.com/ggerganov/llama.cpp/releases
```

**Option 2: Build from source** (for latest features):
```bash
# Clone repository
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp

# Build (CPU only)
make

# Build with Metal (Apple Silicon)
make LLAMA_METAL=1

# Build with CUDA (NVIDIA GPU)
make LLAMA_CUDA=1

# Build with OpenBLAS (faster CPU)
make LLAMA_OPENBLAS=1
```

**Option 3: Python bindings**:
```bash
pip install llama-cpp-python

# With Metal support (Apple Silicon)
CMAKE_ARGS="-DLLAMA_METAL=on" pip install llama-cpp-python

# With CUDA support
CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python
```

In [ ]:
# Check llama.cpp installation
print("Checking llama.cpp Installation")
print("="*60)

# Try command line version
try:
    result = subprocess.run(['llama-cli', '--version'], 
                          capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print("✓ llama.cpp CLI installed")
        print(f"  {result.stdout.strip()}")
    else:
        print("✗ llama.cpp CLI not found")
except Exception:
    print("✗ llama.cpp CLI not found")
    print("  Install: brew install llama.cpp")

# Try Python bindings
try:
    import llama_cpp
    print(f"\n✓ llama-cpp-python installed")
    print(f"  Version: {llama_cpp.__version__}")
except ImportError:
    print("\n✗ llama-cpp-python not installed")
    print("  Install: pip install llama-cpp-python")

print("\n" + "="*60)
print("Note: This notebook demonstrates concepts.")
print("For actual inference, install llama.cpp and download GGUF models.")
print("="*60)

## 4. Converting Models to GGUF

### Using llama.cpp Conversion Script

```bash
# Step 1: Clone llama.cpp
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp

# Step 2: Install requirements
pip install -r requirements.txt

# Step 3: Convert HuggingFace model to GGUF
python convert_hf_to_gguf.py /path/to/model \
    --outfile model-f16.gguf \
    --outtype f16

# Step 4: Quantize to different levels
# Q4_K_M (recommended)
./llama-quantize model-f16.gguf model-q4_k_m.gguf Q4_K_M

# Q5_K_M (higher quality)
./llama-quantize model-f16.gguf model-q5_k_m.gguf Q5_K_M

# Q8_0 (very high quality)
./llama-quantize model-f16.gguf model-q8_0.gguf Q8_0
```

### Or Download Pre-converted Models

**Recommended sources**:
```
TheBloke (HuggingFace):
- 1000+ GGUF models
- All popular models (Llama, Mistral, etc.)
- Multiple quantization levels
- https://huggingface.co/TheBloke

Example: Llama 2 7B
- TheBloke/Llama-2-7B-GGUF
- Q4_K_M: ~4.1GB
- Q5_K_M: ~4.8GB
- Q8_0: ~7.2GB
```

In [ ]:
# Example conversion and usage code
CONVERSION_CODE = '''
# ============= METHOD 1: Convert from HuggingFace =============

# 1. Download model from HuggingFace
from huggingface_hub import snapshot_download
model_path = snapshot_download("meta-llama/Llama-2-7b-hf")

# 2. Convert to GGUF (F16 first)
!python llama.cpp/convert_hf_to_gguf.py {model_path} \\
    --outfile llama-2-7b-f16.gguf \\
    --outtype f16

# 3. Quantize to desired level
!./llama.cpp/llama-quantize llama-2-7b-f16.gguf \\
    llama-2-7b-q4_k_m.gguf Q4_K_M

# ============= METHOD 2: Download pre-converted =============

from huggingface_hub import hf_hub_download

# Download Q4_K_M version
model_path = hf_hub_download(
    repo_id="TheBloke/Llama-2-7B-GGUF",
    filename="llama-2-7b.Q4_K_M.gguf",
    local_dir="./models"
)

print(f"Model downloaded to: {model_path}")

# ============= USAGE: Command Line =============

# Interactive mode
!llama-cli -m models/llama-2-7b.Q4_K_M.gguf \\
    -p "Once upon a time" \\
    -n 100 \\
    --temp 0.7 \\
    --top-k 40 \\
    --top-p 0.9

# Server mode (OpenAI-compatible API)
!llama-server -m models/llama-2-7b.Q4_K_M.gguf \\
    --port 8080 \\
    --ctx-size 2048 \\
    -ngl 0  # 0 = CPU only, >0 = GPU layers

# ============= USAGE: Python Bindings =============

from llama_cpp import Llama

# Load model
llm = Llama(
    model_path="models/llama-2-7b.Q4_K_M.gguf",
    n_ctx=2048,        # Context window
    n_threads=8,       # CPU threads
    n_gpu_layers=0,    # 0 = CPU, >0 = GPU layers (Metal/CUDA)
)

# Generate
output = llm(
    "Once upon a time",
    max_tokens=100,
    temperature=0.7,
    top_p=0.9,
    top_k=40,
    stop=["\n\n"],
)

print(output["choices"][0]["text"])

# Streaming
stream = llm(
    "Once upon a time",
    max_tokens=100,
    stream=True,
)

for chunk in stream:
    text = chunk["choices"][0]["text"]
    print(text, end="", flush=True)
'''

print("GGUF Conversion and Usage Examples")
print("="*60)
print(CONVERSION_CODE)
print("="*60)
print("\nKey Parameters:")
print("  n_ctx: Context window size (512-4096, affects memory)")
print("  n_threads: CPU threads (4-16, more = faster)")
print("  n_gpu_layers: GPU offloading (0=CPU, 35=all layers for 7B)")

## 5. CPU Performance Benchmarks

### Real-World Performance

**Intel/AMD CPUs** (x86):
```
Intel i7-12700K (12 cores):
- Llama 2 7B Q4_0: ~12 tokens/sec
- Llama 2 7B Q8_0: ~8 tokens/sec
- Llama 2 13B Q4_0: ~6 tokens/sec

AMD Ryzen 9 5950X (16 cores):
- Llama 2 7B Q4_0: ~15 tokens/sec
- Llama 2 7B Q8_0: ~10 tokens/sec
- Llama 2 13B Q4_0: ~8 tokens/sec
```

**Apple Silicon** (ARM + Metal):
```
M1 (8GB unified memory):
- Llama 2 7B Q4_0: ~18 tokens/sec
- Llama 2 7B Q8_0: ~12 tokens/sec
- Advantage: Unified memory (no CPU-GPU transfer)

M2 Pro (16GB):
- Llama 2 7B Q4_0: ~28 tokens/sec
- Llama 2 13B Q4_0: ~15 tokens/sec
- Metal acceleration + more bandwidth

M3 Max (64GB):
- Llama 2 7B Q4_0: ~42 tokens/sec
- Llama 2 13B Q4_0: ~22 tokens/sec
- Llama 2 70B Q4_0: ~8 tokens/sec
- Can run 70B models!
```

### Optimization Tips

**1. Choose right quantization**:
- Q4_0/Q4_K_M fastest, good quality
- Q8_0 slower but better quality
- Lower bits = faster

**2. CPU threads**:
- Use physical cores, not hyperthreads
- `n_threads = physical_cores` or `physical_cores - 2`
- Too many threads = slower (context switching)

**3. Memory bandwidth**:
- Fast RAM helps (3200MHz+ DDR4)
- Dual-channel better than single
- Apple Silicon: Unified memory = huge win

**4. Context size**:
- Smaller context = faster (less computation)
- Only use what you need (512-2048 usually enough)
- Large context (4K+) significantly slower

In [ ]:
# Performance comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Platform comparison (Llama 2 7B Q4_0)
ax = axes[0, 0]
platforms = ['i7-12700K\n(x86)', 'Ryzen 9 5950X\n(x86)', 'M1\n(ARM)', 'M2 Pro\n(ARM)', 'M3 Max\n(ARM)']
speeds = [12, 15, 18, 28, 42]
colors_plat = ['#3498db', '#e74c3c', '#95a5a6', '#7f8c8d', '#2c3e50']

bars = ax.barh(platforms, speeds, color=colors_plat, alpha=0.7, edgecolor='black')
ax.set_xlabel('Speed (tokens/sec)', fontsize=12)
ax.set_title('CPU Performance: Llama 2 7B Q4_0', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for bar, speed in zip(bars, speeds):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{speed} tok/s',
            va='center', fontsize=10, fontweight='bold')

# 2. Quantization level impact (M2 Pro)
ax = axes[0, 1]
quant_levels = ['Q2_K', 'Q3_K_M', 'Q4_0', 'Q4_K_M', 'Q5_K_M', 'Q8_0']
speeds_quant = [38, 35, 28, 26, 23, 18]
quality = [4.0, 2.0, 0.8, 0.6, 0.3, 0.1]

# Color by quality
colors_q = plt.cm.RdYlGn_r([q / 4.5 for q in quality])

bars = ax.barh(quant_levels, speeds_quant, color=colors_q, alpha=0.7, edgecolor='black')
ax.set_xlabel('Speed (tokens/sec)', fontsize=12)
ax.set_title('Quantization Impact: Llama 2 7B (M2 Pro)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for bar, speed, qual in zip(bars, speeds_quant, quality):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
            f'{speed} tok/s ({qual:.1f}% loss)',
            va='center', fontsize=9, fontweight='bold')

# 3. Model size impact (M2 Pro, Q4_0)
ax = axes[1, 0]
models = ['7B', '13B', '34B', '70B']
speeds_model = [28, 15, 6, 0]  # 70B doesn't fit on 16GB
ram_needed = [6, 10, 22, 45]

bars = ax.bar(models, speeds_model, alpha=0.7, color='#3498db', edgecolor='black')
ax.set_ylabel('Speed (tokens/sec)', fontsize=12)
ax.set_xlabel('Model Size', fontsize=12)
ax.set_title('Model Size Impact: Q4_0 (M2 Pro 16GB)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add RAM labels
for i, (bar, speed, ram) in enumerate(zip(bars, speeds_model, ram_needed)):
    if speed > 0:
        ax.text(bar.get_x() + bar.get_width()/2, speed + 0.5, 
                f'{speed} tok/s\n({ram}GB RAM)',
                ha='center', fontsize=9, fontweight='bold')
    else:
        ax.text(bar.get_x() + bar.get_width()/2, 1, 
                f'OOM\n({ram}GB needed)',
                ha='center', fontsize=9, fontweight='bold', color='red')

# 4. CPU vs GPU comparison
ax = axes[1, 1]
scenarios = ['7B Q4\nCPU', '7B Q4\nM2 Metal', '7B FP16\nRTX 4090', '70B Q4\nA100']
speeds_hw = [28, 45, 180, 95]
costs = [0, 0, 1500, 10000]  # USD (hardware cost)

# Normalize costs for coloring
colors_cost = plt.cm.Greens_r([c / 10000 for c in costs])

bars = ax.bar(scenarios, speeds_hw, color=colors_cost, alpha=0.7, edgecolor='black')
ax.set_ylabel('Speed (tokens/sec)', fontsize=12)
ax.set_title('CPU vs GPU Performance', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add cost labels
for bar, speed, cost in zip(bars, speeds_hw, costs):
    if cost == 0:
        cost_label = 'Consumer HW'
    else:
        cost_label = f'${cost}'
    ax.text(bar.get_x() + bar.get_width()/2, speed + 5, 
            f'{speed} tok/s\n{cost_label}',
            ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Findings:")
print("1. Apple Silicon 2-3x faster than x86 (unified memory + Metal)")
print("2. Lower quantization = faster (Q4 is 50% faster than Q8)")
print("3. CPU viable for 7B-13B models (5-30 tokens/sec)")
print("4. GPU still 4-6x faster but requires expensive hardware")

## 6. When CPU Inference Makes Sense

### Use CPU/GGUF When:

**✓ Local/Privacy-First Applications**:
- No cloud dependencies
- Data stays on device
- No API costs
- Offline operation

**✓ Consumer Hardware**:
- Laptops without GPUs
- Desktop PCs (no dedicated GPU)
- Apple Silicon Macs (unified memory advantage)
- Cost-sensitive deployments

**✓ Development/Experimentation**:
- Fast iteration
- No GPU setup hassle
- Works anywhere
- Easy sharing (just send GGUF file)

**✓ Moderate Throughput Needs**:
- Personal assistants (1-2 users)
- Interactive applications
- 5-30 tokens/sec acceptable

### Don't Use CPU When:

**✗ High Throughput Required**:
- Production API serving 100+ users
- Batch processing large datasets
- Real-time low-latency (<50ms)

**✗ Very Large Models**:
- 70B+ models on consumer hardware
- Need FP16 precision
- Can't afford quantization loss

**✗ GPU Available**:
- Have NVIDIA GPU → use GPTQ/AWQ + vLLM
- Cloud deployment → GPUs cost-effective at scale

### Cost-Benefit Analysis

```
Scenario 1: Personal Use (1 user)
- CPU (M2 MacBook): $0 (already own), 28 tok/s
- Cloud GPU (OpenAI): $0.002/1K tokens
  → 14K tokens/day = $8.4/month
  → CPU pays for itself immediately

Scenario 2: Small Team (5 users)
- CPU Server ($2K): One-time cost, 15 tok/s
- Cloud GPU (AWS g5): $1.01/hour = $730/month
  → CPU pays for itself in 3 months

Scenario 3: Production (100+ users)
- GPU cluster (A100s): $10-50K + $3K/month
- Better throughput (100+ tok/s)
  → GPU is the right choice
```

In [ ]:
# Decision tree for CPU vs GPU
print("CPU Inference (GGUF) Decision Guide")
print("="*80)
print("""
START: Where to run inference?
  │
  ├─ Do you have a GPU?
  │  ├─ YES (NVIDIA 16GB+) → Consider GPU (GPTQ/AWQ)
  │  │                       But: Try GGUF first for simplicity!
  │  └─ NO → Use GGUF ✓
  │
  ├─ What's your use case?
  │  ├─ Personal/Development → GGUF ✓ (easiest)
  │  ├─ Small team (1-10 users) → GGUF ✓ (cost-effective)
  │  ├─ Production (10-100 users) → GGUF on fast CPU or GPU
  │  └─ Large scale (100+ users) → GPU (vLLM/TensorRT-LLM)
  │
  ├─ What hardware?
  │  ├─ MacBook (M1/M2/M3) → GGUF + Metal ✓ (excellent)
  │  ├─ Laptop (Intel/AMD) → GGUF ✓ (good)
  │  ├─ Desktop (16GB+ RAM) → GGUF ✓ (very good)
  │  └─ Server/Cloud → Consider both (benchmark cost)
  │
  ├─ What model size?
  │  ├─ 7B → GGUF Q4 ✓ (6GB RAM, 10-40 tok/s)
  │  ├─ 13B → GGUF Q4 ✓ (10GB RAM, 5-20 tok/s)
  │  ├─ 33B → GGUF Q4 (22GB RAM, 2-8 tok/s)
  │  └─ 70B → GPU better (45GB RAM, slow on CPU)
  │
  └─ Speed requirements?
     ├─ Interactive (5-10 tok/s) → GGUF ✓
     ├─ Moderate (10-30 tok/s) → GGUF on fast CPU ✓
     ├─ Fast (30-100 tok/s) → GPU
     └─ Very fast (100+ tok/s) → GPU cluster

Recommended Stack:
  Development: GGUF + llama.cpp (fastest to start)
  Production (small): GGUF on dedicated CPU server
  Production (large): GPU + vLLM/TensorRT-LLM
""")

# Use case matrix
print("\nUse Case Matrix")
print("="*80)

use_cases = [
    {'Use Case': 'Personal assistant', 'CPU (GGUF)': 'Excellent', 'GPU': 'Overkill', 'Recommended': 'GGUF'},
    {'Use Case': 'Development/Testing', 'CPU (GGUF)': 'Excellent', 'GPU': 'Faster but complex', 'Recommended': 'GGUF'},
    {'Use Case': 'Privacy-critical', 'CPU (GGUF)': 'Perfect', 'GPU': 'Good (if local)', 'Recommended': 'GGUF'},
    {'Use Case': 'Offline/Edge', 'CPU (GGUF)': 'Only option', 'GPU': 'Not portable', 'Recommended': 'GGUF'},
    {'Use Case': 'Small team API', 'CPU (GGUF)': 'Good', 'GPU': 'Better throughput', 'Recommended': 'Try both'},
    {'Use Case': 'Production API', 'CPU (GGUF)': 'Limited scale', 'GPU': 'Much better', 'Recommended': 'GPU'},
    {'Use Case': 'Batch processing', 'CPU (GGUF)': 'Slow', 'GPU': 'Much faster', 'Recommended': 'GPU'},
    {'Use Case': 'Real-time (<50ms)', 'CPU (GGUF)': 'Challenging', 'GPU': 'Better', 'Recommended': 'GPU'},
]

df = pd.DataFrame(use_cases)
print(df.to_string(index=False))

print("\n" + "="*80)
print("\nBottom Line:")
print("  • GGUF is the best starting point for most people")
print("  • Works everywhere, no GPU needed")
print("  • 7B models run well on consumer hardware (5-40 tok/s)")
print("  • Scale to GPU later if needed")

## Summary

### Key Takeaways

**1. GGUF Format**:
- Optimized for CPU/local inference
- Memory-mapped loading (instant startup)
- Multiple quantization levels (Q2-Q8)
- Cross-platform (x86, ARM, Apple Silicon)

**2. Quantization Levels**:
- Q4_K_M: Best default (4.1GB, 0.6% loss)
- Q5_K_M: Higher quality (4.8GB, 0.3% loss)
- Q8_0: Near-perfect (7.2GB, <0.1% loss)
- Lower bits = smaller + faster but more loss

**3. Performance**:
- Consumer CPUs: 5-15 tokens/sec (7B Q4)
- Apple Silicon: 18-42 tokens/sec (M1-M3)
- Acceptable for interactive use
- 3-6x slower than GPU but no GPU needed

**4. When to Use**:
- Local/privacy-first applications
- Development and experimentation
- Consumer hardware (laptops, desktops)
- Cost-sensitive deployments
- Small-scale production (1-10 users)

**5. Implementation**:
- llama.cpp (C++ implementation)
- Python bindings available
- Pre-converted models (TheBloke)
- Easy to use and deploy

### Comparison with GPU Methods

| Metric | GGUF (CPU) | GPTQ (GPU) | AWQ (GPU) |
|--------|-----------|-----------|----------|
| Hardware | Any CPU | NVIDIA GPU | NVIDIA GPU |
| Speed (7B) | 5-40 tok/s | 100-200 tok/s | 100-200 tok/s |
| Memory | 4-8GB RAM | 4-8GB VRAM | 4-8GB VRAM |
| Setup | Easy | Medium | Medium |
| Cost | $0 (existing HW) | $500-5000 | $500-5000 |
| Quality (INT4) | 0.6-0.8% loss | 1-2% loss | 0.5-1.5% loss |
| Best For | Local, dev, small | Production | Production |

### Best Practices

**Getting Started**:
1. Download pre-converted GGUF model (TheBloke)
2. Start with Q4_K_M (best balance)
3. Use llama.cpp or Python bindings
4. Adjust quantization if needed (Q5_K_M for quality, Q4_0 for speed)

**Optimization**:
1. Use physical CPU cores (not hyperthreads)
2. Enable Metal (Apple Silicon) or OpenBLAS (x86)
3. Minimize context size (only use what you need)
4. Consider RAM speed (faster = better)

**Production**:
1. Use llama-server for API deployment
2. Monitor memory and CPU usage
3. Consider horizontal scaling (multiple instances)
4. Benchmark before committing to CPU vs GPU

### Next Steps

- **Notebook 41**: GPTQ Quantization (GPU alternative)
- **Notebook 42**: AWQ Quantization (GPU alternative)
- **Notebook 56**: Speculative Decoding (speed optimization)
- **LLM_INFERENCE_ROADMAP.md**: Complete optimization roadmap

### Resources

- **llama.cpp**: https://github.com/ggerganov/llama.cpp
- **Python bindings**: https://github.com/abetlen/llama-cpp-python
- **GGUF models**: https://huggingface.co/TheBloke
- **Documentation**: https://github.com/ggerganov/llama.cpp/blob/master/docs/

### The Bottom Line

**GGUF + llama.cpp democratized LLM inference**:
- Run 7B models on any modern laptop
- No GPU required
- Production-viable quality
- Zero cost (use existing hardware)
- Privacy-first (local inference)

**For most developers**: Start with GGUF, scale to GPU if needed.